# CNP on ResNet-50: carton vs. dugong

The paper's watermark experiment (Figure 3) on an ImageNet-pretrained **ResNet-50** instead of VGG-16.
**CNP** (Concept-aware Network Pruning) ranks filters by LRP relevance computed with the watermark concept
subspace ablated, and is compared with **vanilla LRP pruning**.

What differs from the VGG-16 setup:

- **Virtual layer.** It sits after `layer3`, at a stage boundary. That is the only kind of place in a ResNet where
  no skip connection bypasses it (D = 1024).
- **What is pruned.** Only channels inside residual blocks (`conv1`, `conv2` of each bottleneck). The
  residual-stream channels are shared through the additions and stay intact. "Filters pruned (%)" therefore
  refers to these block-internal channels and is not directly comparable to the VGG-16 x-axis.
- **LRP for ranking.** α1β0 with BatchNorm folded into the convs, zB at the stem, and a positive-part split at the
  residual additions (`src/ncp/lrp_resnet.py`).
- **DRSA matrix.** Learned for this network at `layer3` (section 1). The watermark subspace was identified
  from held-out images (sections 2–3) before any pruning run.

Contents:

1. DRSA matrix
2. Which subspace is the watermark?
3. Per-subspace heatmaps
4. Pruning runs: CNP, vanilla, and a control
5. Results

The control is CNP with the ablated virtual layer kept on during fine-tuning. By default CNP uses the ablation
only to rank filters.

**Runtime** (RTX 5080 laptop GPU): about 5 min per pruning run to 80%, and about 11 min for the DRSA
optimization. Every step skips work whose output already exists.

In [ ]:
# ---- Configuration ----
import os
from pathlib import Path

REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
# Images are read from <DATA_ROOT>/images/carton_dugong/original/... (data/images/download_binary_dataset.py).
DATA_ROOT = Path(os.environ.get("NCP_DATA_ROOT", REPO_DIR / "data"))

EXPERIMENT, ARCH, LAYER = "carton", "resnet50", "layer3"
SPURIOUS_SUBSPACE = 3                 # 0-indexed; chosen in section 2
SEEDS = [0, 1, 2, 3, 4]
U_PATH = REPO_DIR / "data" / "projection_matrices" / f"U_{EXPERIMENT}_{ARCH}_{LAYER}.pt"
RESULTS = REPO_DIR / "results"
FIG_DIR = RESULTS / "figures"
DRSA_DIR = RESULTS / "drsa" / f"{EXPERIMENT}_{ARCH}_{LAYER}"

# label -> (pruner, results directory, extra flags)
RUNS = {
    "CNP": ("ncp", RESULTS / "watermark_experiment", []),
    "Vanilla": ("vanilla", RESULTS / "watermark_experiment", []),
    "CNP, fine-tuned with virtual layer": ("ncp", RESULTS / "watermark_experiment_ft_with_virtual",
                                           ["--fine_tune_with_augmented_layers"]),
}
COLORS = {"CNP": "steelblue", "Vanilla": "crimson", "CNP, fine-tuned with virtual layer": "darkorange"}

In [ ]:
import json
import subprocess
import sys

import numpy as np
import torch
from IPython.display import Image, display

sys.path.insert(0, str(REPO_DIR / "src"))
ENV = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src"), "NCP_DATA_ROOT": str(DATA_ROOT), "PYTHONUNBUFFERED": "1"}
LOG_DIR = REPO_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
assert (DATA_ROOT / "images" / "carton_dugong" / "original").exists(), f"no carton/dugong images under {DATA_ROOT}"
print("repo:", REPO_DIR, "| data:", DATA_ROOT, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


def run_script(args, log_name):
    # Run a repo script with the repo's environment. Output goes to logs/<log_name>; its tail is shown on failure.
    log = LOG_DIR / log_name
    with open(log, "w") as f:
        rc = subprocess.run([sys.executable, *map(str, args)], cwd=REPO_DIR, env=ENV,
                            stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print(log.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"{args[0]} failed (exit {rc}); see {log}")

## 1. DRSA matrix at `layer3`

`experiments/compute_drsa_resnet.py` follows the DRSA paper (Chormai et al., TPAMI 2024, Sec. 5 and Suppl. E):

- **Data.** 500 carton training images through the *pretrained* ResNet-50, with activations and LRP context
  vectors at the output of `layer3` taken at 20 random positions per image. The images are carton images
  150–649, which is also the filter-ranking pool; no validation or test images are used.
- **Context.** Computed with Zennit, using LRP-0 (= gradient) above the layer. That mirrors the paper's VGG-16
  setup, where every layer above `conv4_3` uses γ = 0. A `--check` run confirms that the context equals the
  gradient.
- **Fit.** Normalization from Eq. E.1, verified against the objective stored with the authors' published
  matrix. Then `cxai.drsa.optimize` (5000 iterations, best of 3 restarts), giving K = 4 subspaces of 256 dims,
  sorted by mean relevance.

In [ ]:
if not U_PATH.exists():
    run_script(["experiments/compute_drsa_resnet.py", "--experiment", EXPERIMENT, "--arch", ARCH,
                "--layer", LAYER, "--out", U_PATH], f"drsa_{EXPERIMENT}_{ARCH}_{LAYER}.log")
meta = json.loads(U_PATH.with_suffix(".json").read_text())
ext = meta["extract"]
print(f"U: {U_PATH.name}  ({meta['num_subspaces']} subspaces x {meta['subspace_dim']} dims)")
print(f"fit on {ext['n_images']} images x {ext['n_locations']} locations, context w.r.t. logit {ext['target_logit']}, "
      f"gamma={ext['gamma']}; {meta['epochs']} iterations x {meta['trials']} restarts")
print(f"objective {meta['best_objective']:.4f}; orthogonality error {meta['orthogonality_error']:.1e}")
print("mean relevance per subspace on the fitting data (sorted):",
      np.round(meta["subspace_relevance"], 4).tolist())

## 2. Which subspace is the watermark?

The check uses carton test images 800–1017, which are annotated for **natural** watermarks
(`data/images/carton_dugong/test-indices.txt`). Each image's relevance per subspace is computed without and with
the synthetic watermark (`analysis/drsa_watermark_subspace_check.py`).

A watermark subspace should gain relevance from the synthetic watermark on naturally clean images, and also be
higher on naturally watermarked images than on clean ones. Only subspace 3 does both.

In [ ]:
check_path = DRSA_DIR / "watermark_check_natural_test.json"
if not check_path.exists():
    run_script(["analysis/drsa_watermark_subspace_check.py", "--experiment", EXPERIMENT, "--u_path", U_PATH,
                "--arch", ARCH, "--layer", LAYER, "--images", "natural_test"], "drsa_watermark_check.log")
check = json.loads(check_path.read_text())

for group, g in check["groups"].items():
    print(f"\n{group} ({g['n']} images): carton logit {g['logit_without_synthetic']:.2f} -> "
          f"{g['logit_with_synthetic']:.2f} with the synthetic watermark")
    print(f"{'subspace':>8} {'without':>9} {'with':>9} {'change':>14} {'% up':>6}")
    for r in g["subspaces"]:
        print(f"{r['subspace']:>8} {r['mean_without_synthetic']:9.2f} {r['mean_with_synthetic']:9.2f} "
              f"{r['mean_change']:+8.2f} ± {r['stderr_change']:.2f} {100 * r['frac_images_increased']:5.0f}%")

print("\nnatural watermark vs naturally clean (no synthetic watermark):")
for r in check["natural_vs_clean"]:
    print(f"  subspace {r['subspace']}: {r['natural_wm_mean']:7.2f} vs {r['clean_mean']:7.2f}  "
          f"difference {r['difference']:+6.2f} ± {r['stderr']:.2f}")

## 3. Per-subspace heatmaps

The rows show naturally clean carton images without and with the synthetic watermark, then naturally
watermarked images (`analysis/resnet_lrp_subspaces.py`).

- **Forward pass.** Unchanged: the unablated virtual layer is an exact identity.
- **Splitting the relevance.** Relevance is split by subspace at the virtual layer and propagated to the pixels
  separately, so the four subspace heatmaps sum to the standard one.
- **Rules.** LRP-0 (gradient) above the virtual layer, which is how U was fit, and α1β0 below it.
- **Colours.** Red is evidence for carton, blue against.

The second figure shows the same decomposition at the virtual layer itself (14×14, upsampled).

In [ ]:
pixel_fig = FIG_DIR / f"{EXPERIMENT}_{ARCH}_{LAYER}_pixel_subspaces.png"
layer_fig = FIG_DIR / f"{EXPERIMENT}_{ARCH}_{LAYER}_layer_subspaces.png"
if not (pixel_fig.exists() and layer_fig.exists()):
    run_script(["analysis/resnet_lrp_subspaces.py", "--experiment", EXPERIMENT, "--u_path", U_PATH,
                "--arch", ARCH, "--layer", LAYER, "--out_dir", FIG_DIR], "resnet_lrp_subspaces.log")
display(Image(filename=str(pixel_fig), width=900))

In [ ]:
display(Image(filename=str(layer_fig), width=900))

## 4. Pruning runs

The settings are those of `scripts/reproduce_watermark.sh` (paper Table 1):

- 15-epoch head warmup
- LRP-α1β0 ranking on 500 carton training images
- 5% pruning steps up to 80%, with 2 fine-tuning epochs per step and 5 final fine-tuning epochs
- evaluation on the held-out test split after every step

Each run writes `stats.partial.pt` after every evaluation and `stats.pt` when it finishes. Finished runs are
skipped, so the cells below only run what is missing.

**Control.** By default CNP uses the ablated virtual layer only to *rank* filters. It fine-tunes and is
evaluated without it, and the virtual layer is removed before the final fine-tune. The control keeps the ablated
layer on during each step's fine-tuning (`--fine_tune_with_augmented_layers`). That tests whether fine-tuning
under ablation, rather than the choice of filters, explains CNP's robustness.

In [ ]:
COMMON = ["--experiment", EXPERIMENT, "--arch", ARCH, "--virtual_layer", LAYER, "--u_filepath", U_PATH,
          "--spurious_subspace", SPURIOUS_SUBSPACE, "--pr_step", 0.05, "--total_pr", 0.80, "--lr", 1e-4,
          "--momentum", 0.9, "--train_batch_size", 32, "--test_batch_size", 32, "--warmup_epochs", 15,
          "--iter_finetune_epochs", 2, "--final_finetune_epochs", 5, "--rank_loader_type", "positive_only",
          "--eval_on_test"]


def run_dir(label, seed):
    pruner, out, _ = RUNS[label]
    return out / f"{EXPERIMENT}_{ARCH}" / pruner / f"seed{seed}"


def run(label, seed):
    pruner, out, extra = RUNS[label]
    if (run_dir(label, seed) / "stats.pt").exists():
        print(f"[skip] {label}, seed {seed}: results exist")
        return
    print(f"[run] {label}, seed {seed} ...", flush=True)
    tag = label.lower().replace(", ", "_").replace(" ", "_")
    run_script(["experiments/run_watermark_pruning_experiment.py", *COMMON, "--pruner", pruner, "--seed", seed,
                "--out_dir", out, *extra], f"{EXPERIMENT}_{ARCH}_{tag}_seed{seed}.log")

### CNP

In [ ]:
for seed in SEEDS:
    run('CNP', seed)

### Vanilla

In [ ]:
for seed in SEEDS:
    run('Vanilla', seed)

### CNP, fine-tuned with virtual layer

In [ ]:
for seed in SEEDS:
    run('CNP, fine-tuned with virtual layer', seed)

## 5. Results

Mean ± std over seeds (shaded band). Subgroups are named c{class}w{synthetic watermark}: c1 = carton, c0 = dugong.
The key one is **c0w1**, dugongs with the watermark, which a model relying on the watermark calls carton.

In [ ]:
import importlib.util
import matplotlib.pyplot as plt

spec = importlib.util.spec_from_file_location("plot_watermark_results", REPO_DIR / "analysis" / "plot_watermark_results.py")
pw = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pw)
%matplotlib inline
# (after the import: plot_watermark_results.py switches matplotlib to Agg)
N_ITERS = pw.MAX_PRUNE_ITER                      # 16 pruning steps of 5%
SUBGROUPS = {"c1w1": "acc_g1y1", "c1w0": "acc_g0y1", "c0w1": "acc_g1y0", "c0w0": "acc_g0y0"}

data = {}
for label, (pruner, out, extra) in RUNS.items():
    ds = []
    for seed in SEEDS:
        d = pw._load(str(run_dir(label, seed) / "stats.pt"))
        if d is not None:
            ds.append(d)
    for d in ds:
        assert d.get("eval_on_test"), f"{label} seed {d['seed']}: not evaluated on the test split"
        assert d.get("arch") == ARCH and d.get("pruner") == pruner
        if pruner == "ncp":
            # runs from before this was recorded used the default (fine-tune without the virtual layer)
            ft_without = d.get("fine_tune_without_augmented_layers", True)
            assert ft_without == ("--fine_tune_with_augmented_layers" not in extra), f"{label} seed {d['seed']}"
    data[label] = ds
    print(f"{label}: seeds {[d['seed'] for d in ds]}")

In [ ]:
def curve(ds, metric):
    # Mean/std over seeds per pruning step. metric: 'overall' or a subgroup name.
    if metric == "overall":
        dicts = [pw._acc_metric(d) for d in ds]
    else:
        dicts = [pw._scale100(pw._sg_metric(d, SUBGROUPS[metric])) for d in ds]
    niters, means, stds = pw._agg(dicts)
    return pw._niters_to_pct(niters), means, stds


fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
for ax, metric, title in [(axes[0], "overall", "Overall test accuracy"),
                          (axes[1], "c0w1", "c0w1: dugong with watermark")]:
    for label, ds in data.items():
        if ds:
            x, m, s = curve(ds, metric)
            pw._plot_ms(ax, x, m, s, color=COLORS[label], ls="-", label=f"{label} (n={len(ds)})")
    pw._style_ax(ax, ylabel="Accuracy (%)")
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    ax.legend(fontsize=9, loc="lower left")
axes[0].set_xlabel("")
axes[1].set_xlabel("Block-internal channels pruned (%)")
fig.suptitle(f"ResNet-50, {EXPERIMENT} vs dugong: CNP (virtual layer after {LAYER}) vs vanilla LRP pruning")
plt.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / f"{EXPERIMENT}_{ARCH}_accuracy_c0w1.png", dpi=150, bbox_inches="tight")
plt.show()

All four subgroups per method. The **minimum over the subgroups** (bottom row) checks that c0w1 is indeed the
worst one.

In [ ]:
labels = [l for l in RUNS if data[l]]
fig, axes = plt.subplots(2, len(labels), figsize=(5.2 * len(labels), 8), sharey=True, squeeze=False)
styles = {"c1w1": ("steelblue", "--"), "c1w0": ("steelblue", "-"), "c0w1": ("crimson", "--"), "c0w0": ("crimson", "-")}
for j, label in enumerate(labels):
    ax = axes[0, j]
    for sg, (color, ls) in styles.items():
        x, m, s = curve(data[label], sg)
        pw._plot_ms(ax, x, m, s, color=color, ls=ls, label=sg)
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8, loc="lower left")
    ax = axes[1, j]
    mins = [{n: min(pw._sg_metric(d, k).get(n, np.nan) for k in SUBGROUPS.values()) * 100
             for n in pw._acc_metric(d)} for d in data[label]]
    niters, m, s = pw._agg(mins)
    pw._plot_ms(ax, pw._niters_to_pct(niters), m, s, color="black", ls="-", label="min over subgroups")
    ax.legend(fontsize=8, loc="lower left")
for ax in axes.flat:
    pw._style_ax(ax, ylabel="Accuracy (%)")
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / f"{EXPERIMENT}_{ARCH}_subgroups.png", dpi=150, bbox_inches="tight")
plt.show()

### Summary table

The table gives mean ± std over seeds at selected pruning steps and after the final fine-tune ("final"). The
paired comparison counts the seeds in which CNP's c0w1 accuracy beats vanilla's at the same step. Seed k of both
methods uses the same data and splits.

In [ ]:
def at(d, it, metric):
    # Value of a metric at evaluation index `it` (0 = before pruning, N_ITERS + 1 = after the final fine-tune).
    i = d["eval_iter"].index(it)
    if metric == "overall":
        return d["eval_acc"][i]
    return 100 * d["eval_subgroup_stats"][i][SUBGROUPS[metric]]


def ms(vals):
    return f"{np.mean(vals):5.1f} ± {np.std(vals):4.1f}"


steps = [(0, "0%"), (4, "20%"), (8, "40%"), (12, "60%"), (16, "80%"), (N_ITERS + 1, "final")]
for metric in ("overall", "c0w1", "c1w0"):
    print(f"\n{metric} accuracy (%)")
    print(f"{'':36s}" + "".join(f"{name:>15s}" for _, name in steps))
    for label, ds in data.items():
        if ds:
            print(f"{label:36s}" + "".join(f"{ms([at(d, it, metric) for d in ds]):>15s}" for it, _ in steps))

cnp = {d["seed"]: d for d in data["CNP"]}
van = {d["seed"]: d for d in data["Vanilla"]}
paired = sorted(set(cnp) & set(van))
if paired:
    print(f"\nPaired c0w1, CNP - vanilla, seeds {paired}:")
    for it, name in steps:
        diff = [at(cnp[s], it, "c0w1") - at(van[s], it, "c0w1") for s in paired]
        print(f"  {name:>5s}: {np.mean(diff):+6.1f} ± {np.std(diff):4.1f}   CNP higher in {sum(x > 0 for x in diff)}/{len(diff)} seeds")
    steps_all = range(1, N_ITERS + 1)
    wins = sum(at(cnp[s], it, "c0w1") > at(van[s], it, "c0w1") for s in paired for it in steps_all)
    print(f"  all pruning steps (5-80%): CNP higher in {wins}/{len(paired) * N_ITERS} seed-steps")

### Caveats

- **Natural watermarks.** About 27% of the carton test images (59 of 218) already carry a natural watermark.
  So **c1w0** means "no *synthetic* watermark", and c1w1 includes synthetic-on-natural images. The training and
  validation images are not yet annotated, so the share of carton training images showing any watermark is
  somewhat above the nominal 50%. This affects CNP and vanilla equally.
- **The x-axis counts block-internal channels only**, not all filters as in the VGG-16 figures.
- **One DRSA matrix** (DRSA seed 1) is used for all CNP runs. The seeds vary warmup, fine-tuning and data order.
- **The baselines differ slightly between methods within a seed.** Building the virtual layer consumes random
  numbers, so CNP's head initialization differs from vanilla's.